In [7]:
# Install dependencies
!pip install -q streamlit opencv-python-headless numpy pandas matplotlib seaborn

In [ ]:
import os

def convert_y4m_to_raw_yuv(input_file, output_file):
    with open(input_file, 'rb') as f_in:
        # Read the file content
        data = f_in.read()

        # The header starts with 'YUV4MPEG2' and ends with a newline (0x0A)
        # We search for the end of the header
        header_end = data.find(b'\x0a')

        # Strip the header and save the rest as raw YUV
        with open(output_file, 'wb') as f_out:
            f_out.write(data[header_end + 1:])

    print(f"Conversion complete. Raw file saved as: {output_file}")

# Rename your uploaded file to match the input if needed
if os.path.exists('akiyo_qcif.y4m'):
    convert_y4m_to_raw_yuv('akiyo_qcif.y4m', 'akiyo_qcif.yuv')

In [ ]:
import os

# Check if file exists
file_size = os.path.getsize('akiyo_qcif.yuv')
print(f"File size: {file_size} bytes")

if file_size > 0:
    print("Download successful!")
else:
    print("Download still failed.")

In [ ]:
%%writefile intra_analysis.py
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import os

class QualityMetricsCalculator:
    """
    Computes objective quality metrics for video compression analysis.
    """
    @staticmethod
    def compute_metrics(original, reconstructed, qp, standard):
        orig = original.astype(np.float64)
        rec = reconstructed.astype(np.float64)

        # SAD: Sum of Absolute Differences
        sad = np.sum(np.abs(orig - rec))

        # MSE & PSNR
        mse = np.mean((orig - rec) ** 2)
        psnr = 100.0 if mse == 0 else 10 * np.log10((255.0 ** 2) / mse)

        # Estimation of Compression Ratio
        base_bpp = 8.0
        compressed_bpp = base_bpp * (1.0 - (qp / 52.0))
        if standard == "VVC": compressed_bpp *= 0.85
        cr = base_bpp / max(compressed_bpp, 0.1)

        return sad, psnr, cr

class IntraPredictionEngine:
    """
    Core engine to ingest raw YUV data, simulate intra-coding, and generate telemetry.
    """
    @staticmethod
    def process_frame(raw_data, qp=32, grid_size=16):
        width, height = 176, 144
        # Extract Y plane (Luminance)
        gray_frame = np.frombuffer(raw_data, dtype=np.uint8, count=width*height).reshape((height, width))

        # Simulation Logic: Spatial Gradient Analysis
        grad_x = cv2.Sobel(gray_frame, cv2.CV_64F, 1, 0, ksize=3)
        grad_y = cv2.Sobel(gray_frame, cv2.CV_64F, 0, 1, ksize=3)
        magnitude = cv2.magnitude(grad_x, grad_y)

        reconstructed = np.zeros_like(gray_frame)
        grid_modes = np.zeros((grid_size, grid_size), dtype=np.int32)

        block_h, block_w = height // grid_size, width // grid_size

        for i in range(grid_size):
            for j in range(grid_size):
                ys, ye = i * block_h, (i+1) * block_h
                xs, xe = j * block_w, (j+1) * block_w
                var = np.mean(magnitude[ys:ye, xs:xe])

                # Mode selection heuristic
                selected_idx = 1 if var < 15 else (2 if np.mean(grad_x[ys:ye, xs:xe]) > 0 else 3)
                grid_modes[i, j] = selected_idx

                # Simulated Quantization Noise
                noise = np.random.normal(0, (qp / 52.0) * 30.0, (block_h, block_w))
                reconstructed[ys:ye, xs:xe] = np.clip(gray_frame[ys:ye, xs:xe] + noise, 0, 255)

        return gray_frame, reconstructed, grid_modes

    @staticmethod
    def stream_frames(yuv_path, num_frames=300):
        width, height = 176, 144
        # Calculate size for YUV 4:2:0: Y(W*H) + U(W/2*H/2) + V(W/2*H/2)
        # = W*H + 0.25*W*H + 0.25*W*H = 1.5 * W * H
        frame_size = int(width * height * 1.5)

        with open(yuv_path, 'rb') as f:
            for _ in range(num_frames):
                raw = f.read(frame_size)
                # If we read fewer bytes than expected, the file might have ended
                if len(raw) < frame_size:
                    print(f"Warning: Reached end of file or incomplete frame. Read {len(raw)} bytes.")
                    break
                yield raw

if __name__ == "__main__":
    yuv_file = "akiyo_qcif.yuv"
    all_results = []

    print("Starting full sequence analysis...")
    frame_gen = IntraPredictionEngine.stream_frames(yuv_file, num_frames=300)

    for idx, raw_frame in enumerate(frame_gen):
        orig, rec, grid = IntraPredictionEngine.process_frame(raw_frame)
        sad, psnr, cr = QualityMetricsCalculator.compute_metrics(orig, rec, 32, "VVC")
        all_results.append({"sad": sad, "psnr": psnr, "cr": cr})

        if (idx + 1) % 50 == 0:
            print(f"Processed frame {idx + 1}/300")

    # Aggregate and Report
    avg_psnr = np.mean([r['psnr'] for r in all_results])
    avg_sad = np.mean([r['sad'] for r in all_results])
    avg_cr = np.mean([r['cr'] for r in all_results])

    print(f"\nAnalysis Complete.")
    print(f"Average PSNR: {avg_psnr:.2f} dB | Average SAD: {avg_sad:.0f} | Compression Ratio: {cr:.2f} : 1")

    # Visualization using the last processed frame
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    ax[0].imshow(orig, cmap='gray'); ax[0].set_title("Original (Final Frame)"); ax[0].set_xlabel("Width (px)"); ax[0].set_ylabel("Height (px)")
    ax[1].imshow(rec, cmap='gray'); ax[1].set_title("Reconstructed (Final Frame)"); ax[1].set_xlabel("Width (px)")
    sns.heatmap(grid, ax=ax[2], cmap="plasma", xticklabels=range(0, 16), yticklabels=range(0, 16))
    ax[2].set_title("Visualization Heatmap"); ax[2].set_xlabel("Block Index (X)"); ax[2].set_ylabel("Block Index (Y)")

    plt.tight_layout()
    plt.savefig("analysis_report.png")
    print("Report saved as 'analysis_report.png'.")

In [ ]:
!python intra_analysis.py